# Pipeline de tri de pages financières (PDF normaux + scannés)

Ce notebook orchestre un pipeline **modulaire** composé de fichiers Python séparés :

| Fichier | Rôle |
|---|---|
| `config.py` | Mots-clés EN/FR, poids, seuils — **modifiez les règles ici** |
| `text_extraction.py` | Extraction de texte (natif / OCR / hybride) — **remplacez l'OCR ici** |
| `relevance.py` | Scoring de pertinence (mots-clés, densité numérique, zero-shot optionnel) |
| `pdf_processor.py` | Traite un PDF ou un dossier entier, écrit les PDF filtrés |
| `report.py` | Génère le fichier Excel global |

**Pourquoi c'est modulaire ?** Chaque brique dépend d'une interface abstraite
(`TextExtractor`, `RelevanceScorer`) et non d'une implémentation précise.
Par exemple, si demain vous voulez remplacer Tesseract par EasyOCR ou une API
cloud, il suffit d'écrire une nouvelle classe héritant de `TextExtractor` dans
`text_extraction.py` — **aucune autre partie du pipeline n'a besoin de changer**.


## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # s'assure que les modules du dossier sont importables

from text_extraction import build_extractor, NativeTextExtractor, TesseractOCRExtractor, HybridTextExtractor
from relevance import build_default_composite_scorer, KeywordScorer, NumericDensityScorer, CompositeScorer
from pdf_processor import process_folder
from report import write_excel_report
import config


## 2. Paramètres

- `INPUT_DIR` : dossier contenant les PDF financiers à traiter (normaux ou scannés)
- `OUTPUT_DIR` : dossier où seront écrits les PDF filtrés (mêmes noms de fichiers, mais seulement les pages pertinentes)
- `EXCEL_REPORT_PATH` : chemin du fichier Excel récapitulatif global (situé **en dehors** de `OUTPUT_DIR`)


In [ ]:
INPUT_DIR = "input_pdfs"                 # <-- dossier avec vos PDF financiers
OUTPUT_DIR = "output_pdfs"               # <-- dossier de sortie (PDF filtrés)
EXCEL_REPORT_PATH = "rapport_global.xlsx"  # <-- fichier Excel global (hors OUTPUT_DIR)

os.makedirs(INPUT_DIR, exist_ok=True)
print(f"Déposez vos PDF dans le dossier: {os.path.abspath(INPUT_DIR)}")


## 3. Configuration des règles de détection (optionnel)

Toutes les valeurs ci-dessous viennent de `config.py`. Vous pouvez soit éditer
directement `config.py`, soit surcharger certains paramètres ici sans toucher
au fichier (utile pour tester rapidement différents seuils).

In [ ]:
settings = dict(config.DEFAULT_SETTINGS)  # copie modifiable

# Exemples de réglages que vous pouvez ajuster :
# settings["extraction_mode"] = "ocr_only"      # force l'OCR sur toutes les pages
# settings["confirmation_threshold"] = 0.30      # confirmation plus permissive
# settings["numeric_density_threshold"] = 0.05
# settings["require_keyword_match"] = True       # revient au mode strict
# settings["enable_continuation_detection"] = False
# settings["use_zero_shot"] = True               # nécessite: pip install transformers torch

# --- gros volumes (beaucoup de PDF / documents longs) ---
# import os
# settings["parallel_workers"] = os.cpu_count() - 1  # traitement en parallèle
# settings["enable_checkpoint"] = True                # reprise après plantage (activé par défaut)
# settings["retry_errors"] = True                     # retente les fichiers en erreur au run suivant

settings


## 4. Construction des modules (extracteur de texte + scorer de pertinence)

C'est ICI que vous changeriez de moteur OCR ou de stratégie d'extraction si
besoin. `build_extractor(settings)` choisit automatiquement l'implémentation
selon `settings["extraction_mode"]` :
- `"hybrid"` (par défaut) : texte natif, bascule sur l'OCR si trop court ou
  si l'encodage semble corrompu.
- `"ocr_only"` : force l'OCR sur TOUTES les pages, natif ou pas. À utiliser
  si le mode hybride lit mal vos documents (problème d'encodage de police).
- `"native_only"` : jamais d'OCR, uniquement le texte déjà encodé dans le PDF.

Pour remplacer complètement le moteur OCR (Tesseract -> autre chose), créez
une nouvelle classe héritant de `TextExtractor` dans `text_extraction.py`,
puis adaptez `build_extractor()` pour la retourner. Le reste du pipeline ne
change pas.

In [ ]:
# --- Extraction de texte : choisie automatiquement selon settings["extraction_mode"] ---
extractor = build_extractor(settings)

# --- Scoring de pertinence (mots-clés + densité numérique + zero-shot optionnel) ---
scorer = build_default_composite_scorer(settings)

print("Mode d'extraction:", settings["extraction_mode"])
print("Extracteur:", extractor)
print("Scorer:", scorer)


## 5. Exécution du pipeline sur le dossier

In [ ]:
results = process_folder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    extractor=extractor,
    scorer=scorer,
    settings=settings,
    verbose=True,
)


## 6. Génération du rapport Excel global

In [ ]:
write_excel_report(results, EXCEL_REPORT_PATH)
print(f"PDF filtrés dans: {os.path.abspath(OUTPUT_DIR)}")
print(f"Rapport Excel: {os.path.abspath(EXCEL_REPORT_PATH)}")


## 7. Aperçu rapide des résultats dans le notebook

In [ ]:
from report import build_summary_dataframe, build_detail_dataframe

summary_df = build_summary_dataframe(results)
summary_df


In [ ]:
detail_df = build_detail_dataframe(results)
detail_df


## 8. (Optionnel) Tester la détection sur un seul texte

Pratique pour ajuster vos mots-clés dans `config.py` sans relancer tout le dossier.

In [ ]:
sample_text = """
Consolidated Statement of Financial Position
Total assets 1 234 567   Total liabilities 654 321
"""

evaluation = scorer.evaluate(sample_text)
print("Pertinent ?", evaluation.is_relevant)
print("Score:", evaluation.score)
print("Mots-clés trouvés:", evaluation.matched_keywords)
print("Détail:", evaluation.details)


---
### Comment la pertinence est décidée (règles actuelles)

Pour chaque page, dans l'ordre :

1. On cherche un mot-clé de `config.py` (ex: "bilan", "balance sheet"...).
2. **Dans tous les cas** (mot-clé trouvé ou non), on calcule la densité
   numérique de la page (proportion de chiffres + volume absolu de chiffres).
3. Décision :
   - **Mot-clé trouvé + densité suffisante** → page gardée
   - **Mot-clé trouvé + densité insuffisante** → page rejetée (ex: le mot
     "bilan" apparaît juste dans un sommaire ou une phrase narrative, sans
     le tableau chiffré)
   - **Pas de mot-clé + densité suffisante** → page gardée quand même (ex:
     une page qui continue un tableau financier sur plusieurs pages, sans
     répéter le titre)
   - **Pas de mot-clé + densité insuffisante** → page rejetée

En clair : **la densité numérique est toujours vérifiée**, que le mot-clé
matche ou non. Le mot-clé seul ne garde jamais une page, et son absence ne
la rejette pas non plus si les chiffres sont suffisants.

**Mode strict optionnel** : passez `settings["require_keyword_match"] = True`
pour revenir à un mode plus conservateur où une page SANS mot-clé est
toujours rejetée, quels que soient les chiffres.

**Détection de continuation** (filet de sécurité supplémentaire, activé par
défaut) : en plus de la règle ci-dessus, une page sans mot-clé qui suit
immédiatement une page pertinente est acceptée avec un seuil de densité
légèrement différent (`continuation_threshold`).

### Notes sur la modularité

- **Changer d'OCR ou de stratégie d'extraction** : `settings["extraction_mode"]`
  accepte `"hybrid"` (par défaut), `"ocr_only"` (force l'OCR sur toutes les
  pages — à utiliser si le mode hybride lit mal vos documents à cause d'un
  problème d'encodage de police natif), ou `"native_only"`. Pour changer
  complètement de moteur OCR (Tesseract -> autre chose), créez une nouvelle
  classe héritant de `TextExtractor` dans `text_extraction.py` et adaptez
  `build_extractor()`. Rien d'autre à modifier.
- **Changer/ajouter des mots-clés** : éditez uniquement `config.py`
  (`KEYWORDS_EN`, `KEYWORDS_FR`).
- **Activer le zero-shot** : `pip install transformers torch`, puis
  `settings["use_zero_shot"] = True` dans la section 3. Le modèle par défaut
  (`joeddav/xlm-roberta-large-xnli`) est multilingue (EN/FR).
- **Ajuster l'exigence de densité numérique** : modifiez
  `numeric_density_threshold` / `numeric_min_digit_count` (richesse
  numérique brute) et `confirmation_threshold` (seuil final) dans
  `config.py`. Utilisez `density_tester.py` (voir section 9 ci-dessous)
  pour observer l'effet de ces réglages sur vos propres documents avant de
  relancer tout le pipeline.
- **Ajuster la détection de continuation** : `continuation_threshold`, ou
  `enable_continuation_detection = False` pour la désactiver.

### Gros volumes : reprise après plantage et traitement en parallèle

- **Reprise automatique** : `enable_checkpoint = True` (par défaut) écrit
  une sauvegarde (`<output_dir>/.pipeline_checkpoint.json`) après chaque
  PDF traité. Si le pipeline plante ou est interrompu au milieu d'un gros
  lot, relancez simplement la même cellule : les PDF déjà traités avec
  succès sont automatiquement ignorés, seuls les fichiers restants (et ceux
  qui avaient échoué) sont retraités.
- **Traitement en parallèle** : `settings["parallel_workers"] = N` traite N
  PDF simultanément sur plusieurs cœurs CPU. Recommandé de rester en
  séquentiel (`N=1`, valeur par défaut) tant que vous validez les réglages
  sur vos documents, puis d'augmenter une fois satisfait du résultat.
  Incompatible avec `use_zero_shot=True`.
- **Fichiers en erreur** : `retry_errors = True` (par défaut) fait que tout
  fichier ayant échoué à un run précédent est automatiquement retenté au
  run suivant (utile si vous corrigez/remplacez le fichier source) ; seuls
  les fichiers traités avec succès restent définitivement ignorés lors
  d'une reprise.


## 9. (Optionnel) Tester la densité numérique d'un PDF page par page

`density_tester.py` réutilise EXACTEMENT le même code d'extraction que le
pipeline (`build_extractor`), et affiche pour chaque page : la méthode
d'extraction utilisée, le nombre de chiffres, la densité (ratio), le score
normalisé, et si elle dépasse le seuil de confirmation actuel.

Pratique pour calibrer `numeric_density_threshold`, `numeric_min_digit_count`
et `confirmation_threshold` sur vos propres documents, sans avoir à relancer
tout le pipeline de filtrage à chaque essai. Voir aussi la section
"Tests unitaires" du module dans `modules.ipynb` pour des tests plus poussés.

In [ ]:
from density_tester import analyze_pdf_density
from IPython.display import display

# Remplacez par le chemin d'un de vos PDF (ou un fichier de input_pdfs/)
test_pdf_path = os.path.join(INPUT_DIR, os.listdir(INPUT_DIR)[0]) if os.listdir(INPUT_DIR) else None

if test_pdf_path:
    density_df = analyze_pdf_density(test_pdf_path, settings)
    print(f"Analyse de: {test_pdf_path}")
    display(density_df)
else:
    print(f"Aucun PDF trouvé dans {INPUT_DIR}")
